In [ ]:
import pandas as pd

# Define South Yorkshire regions
south_yorkshire = ["Barnsley", "Doncaster", "Rotherham", "Sheffield"]

# Filter the dataframe
df_sy = df_test[df_test["GEOGRAPHY_NAME"].isin(south_yorkshire)].reset_index(drop=True)

print(df_sy)

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# --- 2. Build composite y via PCA on BUSINESSES & EMPLOYEES ---
y_raw = df_sy[["BUSINESSES", "EMPLOYEES"]]

y_scaler = StandardScaler()
y_scaled = y_scaler.fit_transform(y_raw)

pca_y = PCA(n_components=1)
y_composite = pca_y.fit_transform(y_scaled).flatten()
y = pd.Series(y_composite, index=df_sy.index, name="y_composite")

# --- 3. Diagnostics ---
print(f"Explained variance by composite: {pca_y.explained_variance_ratio_[0]*100:.1f}%")
print(f"Loadings — BUSINESSES: {pca_y.components_[0][0]:.3f} | EMPLOYEES: {pca_y.components_[0][1]:.3f}")
print("\ny_composite summary:")
print(y.describe())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# --- Assumes these are already built from the PCA script ---
# df_sy        : filtered South Yorkshire dataframe
# y            : y_composite series (same index as df_sy)
# scores_df    : PC scores dataframe with all pc_cols + GEOGRAPHY_NAME (from PCA script)
# pc_cols      : list of PC column names

# --- 1. Attach locality, year and y_composite to scores ---
df_analysis = scores_df.copy()
df_analysis["GEOGRAPHY_NAME"] = df_sy["GEOGRAPHY_NAME"].values
df_analysis["YEAR"] = df_sy["YEAR"].values          # <-- adjust column name if needed
df_analysis["y_composite"] = y.values

# --- 2. Year-over-year change in y_composite per locality ---
df_analysis = df_analysis.sort_values(["GEOGRAPHY_NAME", "YEAR"])
df_analysis["y_growth"] = df_analysis.groupby("GEOGRAPHY_NAME")["y_composite"].diff()

# --- 3. Mean PC scores per locality ---
pc_by_locality = df_analysis.groupby("GEOGRAPHY_NAME")[pc_cols].mean().reset_index()
pc_by_locality.columns = ["GEOGRAPHY_NAME"] + [f"mean_{c}" for c in pc_cols]

# --- 4. Mean YoY growth per locality ---
growth_by_locality = (
    df_analysis.groupby("GEOGRAPHY_NAME")["y_growth"]
    .mean()
    .reset_index()
    .rename(columns={"y_growth": "mean_y_growth"})
)

# --- 5. Merge ---
summary = pc_by_locality.merge(growth_by_locality, on="GEOGRAPHY_NAME")
print(summary)

# --- 6. Plot: mean PC score vs mean y growth per locality ---
mean_pc_cols = [f"mean_{c}" for c in pc_cols]
n_pcs = len(mean_pc_cols)
ncols = 3
nrows = -(-n_pcs // ncols)  # ceiling division

fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 4))
axes = axes.flatten()

for i, pc in enumerate(mean_pc_cols):
    ax = axes[i]
    for _, row in summary.iterrows():
        ax.scatter(row[pc], row["mean_y_growth"], s=100)
        ax.annotate(row["GEOGRAPHY_NAME"], (row[pc], row["mean_y_growth"]),
                    fontsize=8, textcoords="offset points", xytext=(5, 3))
    ax.axhline(0, color="grey", linestyle="--", linewidth=0.8)
    ax.axvline(0, color="grey", linestyle="--", linewidth=0.8)
    ax.set_xlabel(pc.replace("mean_", ""), fontsize=9)
    ax.set_ylabel("Mean YoY y_composite growth", fontsize=9)
    ax.set_title(f"{pc} vs growth", fontsize=10)

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Mean PC Scores vs YoY Growth in y_composite — South Yorkshire", fontsize=12)
plt.tight_layout()
plt.show()

# --- 7. Export ---
summary.to_excel("growth_vs_pca_locality.xlsx", index=False)
print("Exported to growth_vs_pca_locality.xlsx")